In [1]:
import pandas as pd
import numpy as np

from cosmic.sample.initialbinarytable import InitialBinaryTable
from cosmic.sample.sampler import independent
from cosmic.evolve import Evolve

from ccsnlab.data_loading.process_raw import create_sn_info
from ccsnlab.data_loading.rerun_binary import evolve_and_save

Create a small population with cosmic

In [4]:
final_kstars = np.linspace(0, 14, 15)
Z = 0.02
binfrac = 'offner23'

InitialBinaries, mass_singles, mass_binaries, n_singles, n_binaries = InitialBinaryTable.sampler(
    'independent', final_kstars, final_kstars,
    binfrac_model='offner23', primary_model='kroupa01',
    ecc_model='sana12', porb_model='martinez26',
    qmin=-1, SF_start=13700.0, SF_duration=0.0,
    met=0.02, size=int(1e3)
)

#only evolve binaries massive enough (or close to massive enough) to create at least 1 CCSN
InitialBinaries = InitialBinaries[InitialBinaries.mass_1 + InitialBinaries.mass_2 > 6.0]

Evolve the population

In [5]:
BSEDict = {
    "pts1": 0.001, "pts2": 0.01, "pts3": 0.02, "zsun": 0.014, "windflag": 3,
    "eddlimflag": 0, "neta": 0.5, "bwind": 0.0, "hewind": 0.5, "beta": 0.125,
    "xi": 0.5, "acc2": 1.5, "LBV_flag": 1, "alpha1": 1.0, "lambdaf": 0.0,
    "ceflag": 1, "cekickflag": 2, "cemergeflag": 1, "cehestarflag": 0,
    "qcflag": 5,
    "qcrit_array": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
    "kickflag": 5, "sigma": 265.0, "bhflag": 1, "bhsigmafrac": 1.0,
    "sigmadiv": -20.0, "ecsn": 2.25, "ecsn_mlow": 1.6, "aic": 1, "ussn": 1,
    "polar_kick_angle": 90.0,
    "natal_kick_array": [[-100.0, -100.0, -100.0, -100.0, 0.0], [-100.0, -100.0, -100.0, -100.0, 0.0]],
    "mm_mu_ns": 400.0, "mm_mu_bh": 200.0, "remnantflag": 4,
    "fryer_mass_limit": 0, "mxns": 3.0, "rembar_massloss": 0.5,
    "wd_mass_lim": 1, "maltsev_mode": 0, "maltsev_fallback": 0.5,
    "maltsev_pf_prob": 0.1, "pisn": -2, "ppi_co_shift": 0.0,
    "ppi_extra_ml": 0.0, "bhspinflag": 0, "bhspinmag": 0.0, "grflag": 1,
    "eddfac": 10, "gamma": -2, "don_lim": -1, "acc_lim": -1, "tflag": 1,
    "ST_tide": 1,
    "fprimc_array": [2.0/21.0,2.0/21.0,2.0/21.0,2.0/21.0,2.0/21.0,2.0/21.0,2.0/21.0,2.0/21.0,2.0/21.0,2.0/21.0,2.0/21.0,2.0/21.0,2.0/21.0,2.0/21.0,2.0/21.0,2.0/21.0],
    "ifflag": 1, "wdflag": 1, "epsnov": 0.001, "bdecayfac": 1,
    "bconst": 3000, "ck": 1000, "rejuv_fac": 1.0, "rejuvflag": 0,
    "bhms_coll_flag": 0, "htpmb": 1, "ST_cr": 1, "rtmsflag": 0
}

# This just wraps COSMIC's evolve.evolve, and uses high bcm resolution to catch IIns 
bpp, bcm, _, _ = evolve_and_save(InitialBinaries, BSEDict)

Compile a sn_info dataframe!

In [8]:
sample_mass = mass_singles + mass_binaries
singles_mass = mass_singles
n_stars = n_singles + n_binaries

sn_info = create_sn_info(bpp,
                         bcm,
                         Z,
                         BSEDict,
                         binfrac,
                         sample_mass,
                         singles_mass,
                         n_stars,
                         n_singles
                        )

In [9]:
print(sn_info)

    bin_num  SN_1  SN_2 merger_type  zams_mass_1  zams_mass_2     zams_porb  \
0         0     1     1        -001    27.102089    10.477613  2.909447e+03   
1         1     0     0        -001     4.591566     2.019607  4.932028e+06   
2         2     0     0        -001     4.655261     4.572574  1.027581e+05   
3         3     0     0        0801     8.675849     6.922672  7.772675e+01   
4         4     0     0        -001     4.696882     3.345560  1.002287e+05   
..      ...   ...   ...         ...          ...          ...           ...   
62       68     0     0        -001     3.748426     2.706565  2.246277e+08   
63       69     0     0        -001     4.095824     3.191419  2.086356e+06   
64       70     0     0        -001     5.580587     2.911017  1.372501e+04   
65       71     0     0        -001     6.274285     4.431255  1.118733e+05   
66       72     0     0        0301     4.053824     3.826442  9.646456e+01   

    zams_ecc      zams_sep  sn_1_time  ...  maltsev